# SVD - Low Rank Approximations


# Low Rank Approximations

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/12-SVD-Low-Rank.ipynb)

We now consider applications of the Singular Value Decomposition (SVD).

> SVD is "the Swiss Army Knife of Numerical Linear Algebra.”

Dianne O’Leary, MMDS ’06 (Workshop on Algorithms for Modern Massive Data Sets)

## Lecture Organization

In this lecture we:

* review the linear algebra the SVD is built on: orthonormal vectors, eigenvalues of symmetric matrices, and matrix rank;
* define the SVD, see where its three factors come from, and work one out by hand;
* prove what it is good for: the SVD gives the best __low rank approximation__ to any matrix (the Eckart–Young theorem);
* apply it to data matrices from four applications
    * internet traffic data
    * social media data
    * image data
    * movie data


The other big use of the SVD, __dimensionality reduction__ (PCA), is the subject of the next lecture.


# Linear Algebra Prerequisites

## Notation

Throughout this lecture a matrix $A \in \mathbb{R}^{m\times n}$ has $m$ rows and $n$ columns, and we assume $m > n$: more rows than columns.

For a data matrix this means rows are observations and columns are features, with more observations than features — the common case.


Note that this is the matrix-theory convention. In the regression lectures we wrote the data matrix as $n \times p$ with $n$ observations; here $m$ plays the role of $n$ and $n$ the role of $p$.


The next few slides summarize what we need from the [Linear Algebra Refresher](A7-Linear-Algebra-Refresher.qmd) — see its sections on [orthogonal matrices](A7-Linear-Algebra-Refresher.qmd#orthogonal-matrices), [eigenvalues and eigenvectors](A7-Linear-Algebra-Refresher.qmd#eigenvalues-and-eigenvectors) and the [eigendecomposition](A7-Linear-Algebra-Refresher.qmd#eigendecomposition) for more detail and examples.


## Orthonormal Vectors and Orthogonal Matrices

A set of vectors $\{\mathbf{q}_1, \dots, \mathbf{q}_n\}$ is __orthonormal__ if the vectors are mutually perpendicular and each has unit length:

$$
\mathbf{q}_i^T \mathbf{q}_j = \begin{cases} 1 & i = j \\ 0 & i \neq j. \end{cases}
$$

Collect them as the columns of a matrix $Q$. Then the entries of $Q^TQ$ are exactly these dot products, so

$$
Q^T Q = I.
$$

If $Q$ is also square, then $Q Q^T = I$ as well: $Q$ is an __orthogonal matrix__ and its inverse is simply its transpose, $Q^{-1} = Q^T$.


Orthogonal matrices preserve lengths and angles: $\Vert Q\mathbf{x}\Vert = \Vert\mathbf{x}\Vert$. Geometrically they are rotations and reflections. A familiar example is the $2\times 2$ rotation matrix

$$
R(\theta) = \begin{bmatrix} \cos \theta & -\sin \theta\\ \sin \theta & \cos \theta\end{bmatrix}, \qquad R(\theta)^T R(\theta) = I.
$$


The terminology is unfortunate but standard: an *orthogonal matrix* has *orthonormal* columns.


## Eigenvalues and Eigenvectors

For a __square__ matrix $S \in \mathbb{R}^{n\times n}$, a nonzero vector $\mathbf{x}$ is an __eigenvector__ with __eigenvalue__ $\lambda$ if

$$
S\mathbf{x} = \lambda \mathbf{x}.
$$

$S$ does not change the direction of $\mathbf{x}$; it only stretches it by the factor $\lambda$.

For example,

$$
S = \begin{bmatrix}2 & 1\\1 & 2\end{bmatrix}
$$

has eigenvalues $\lambda_1 = 3$ and $\lambda_2 = 1$, with eigenvectors $\mathbf{x}_1 = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}$ and $\mathbf{x}_2 = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix}$.

Check: $S\mathbf{x}_1 = \frac{1}{\sqrt{2}}\begin{bmatrix}3\\3\end{bmatrix} = 3\mathbf{x}_1$ and $S\mathbf{x}_2 = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix} = 1\cdot\mathbf{x}_2$.


## Symmetric Matrices

The matrix $S$ above is __symmetric__: $S = S^T$. Symmetric matrices have two properties we will rely on (this is the *spectral theorem*):

* all $n$ eigenvalues are real, and
* the eigenvectors can be chosen to be __orthonormal__.


Collecting the eigenvectors into an orthogonal matrix $Q$ and the eigenvalues into a diagonal matrix $\Lambda$ gives the __eigendecomposition__

$$
S = Q \Lambda Q^T.
$$

For the example above, $Q = \frac{1}{\sqrt{2}}\begin{bmatrix}1 & 1\\ 1 & -1\end{bmatrix}$ and $\Lambda = \begin{bmatrix}3 & 0\\ 0 & 1\end{bmatrix}$. Notice that the two eigenvectors are indeed perpendicular.


One more fact. If $S = B^TB$ for some matrix $B$, then every eigenvalue is __nonnegative__, because for a unit eigenvector $\mathbf{x}$

$$
\lambda = \mathbf{x}^T S \mathbf{x} = \mathbf{x}^T B^T B \mathbf{x} = \Vert B\mathbf{x}\Vert^2 \geq 0.
$$

Keep $B^TB$ in mind — it is about to reappear as $A^TA$.


## Matrix Rank

Let's review some definitions.


Let $A\in\mathbb{R}^{m\times n}$ be a real matrix with $m>n$.

The __rank__ of $A$ is the number of linearly independent rows or columns of the matrix.


The largest value that a matrix rank can take is $\min(m,n)$. Since we assumed $m>n$, the largest value of the rank is $n$.


If the matrix $A$ has rank equal to $n$, then we say it is **full rank**.


However, it can happen that the rank of a matrix is __less__ than $\min(m,n)$. In this case we say that $A$ is **rank-deficient**.


## Matrix Rank and Column Space

The dimension of the column space of $A$ is the **smallest number of vectors that suffice to construct the columns of $A$.**

And it is equal to the rank of the matrix.

If the dimension of the column space is $k$, then there exists a set of vectors $\{\mathbf{c}_1, \mathbf{c}_2, \dots, \mathbf{c}_k\}$ such that every column $\mathbf{a}_i$ of $A$ can be expressed as:

$$\mathbf{a}_i = r_{1i}\mathbf{c}_1 + r_{2i}\mathbf{c}_2 + \dots + r_{ki}\mathbf{c}_k\quad i=1,\ldots,n.$$


The vectors $\mathbf{x}$ with $A\mathbf{x} = \mathbf{0}$ form the __null space__ of $A$. When $A$ has rank $k$, the null space has dimension $n - k$.


---

To store a matrix $A \in \mathbb{R}^{m\times n}$ we need to store $mn$ values.

However, if $A$ has rank $k$, it can be factored as $A = CR$,
$$
A =
\begin{bmatrix}
\bigg| & \bigg| &   & \bigg| \\
\mathbf{c}_1   & \mathbf{c}_2  & \dots  & \mathbf{c}_k  \\
\bigg| & \bigg| &  & \bigg|
\end{bmatrix}
\begin{bmatrix}
\vert & \vert &   & \vert \\
\mathbf{r}_1   & \mathbf{r}_2  & \dots  & \mathbf{r}_n  \\
\vert & \vert &  & \vert
\end{bmatrix}
$$

where $C \in \mathbb{R}^{m\times k}$ and $R \in \mathbb{R}^{k \times n}$.

This requires $k(m+n)$ values, which could be much smaller than $mn$ when $k$ is small, e.g. $k < \frac{mn}{m+n}$.

# The Singular Value Decomposition

## Singular Vectors and Values

For

$$
A\in\mathbb{R}^{m\times n} \text{ with } m>n \text{ and rank } k,
$$

there exists an orthonormal set of vectors $\{\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_n\}$ with $\mathbf{v}_i \in \mathbb{R}^n$

and an orthonormal set of vectors $\{\mathbf{u}_1, \mathbf{u}_2, \dots, \mathbf{u}_n\}$ with $\mathbf{u}_i \in \mathbb{R}^m$

such that

$$
A\mathbf{v}_1 = \sigma_1 \mathbf{u}_1 \quad\cdots\quad A\mathbf{v}_k = \sigma_k \mathbf{u}_k, \quad A\mathbf{v}_{k+1} = 0 \quad\cdots\quad A\mathbf{v}_n = 0.
$$

where $\sigma_1 \geq \sigma_2 \geq \dots \geq \sigma_k > 0$ are the _singular values_ of $A$.

* $\mathbf{v}_{k+1}, \ldots, \mathbf{v}_n$ are an orthonormal basis for the null space of $A$, and
* $\mathbf{u}_{k+1}, \ldots, \mathbf{u}_n$ are any unit vectors orthogonal to $\mathbf{u}_1, \ldots, \mathbf{u}_k$ (and to each other). They are multiplied by zero, so their choice does not matter.


Compare with an eigenvector: $A$ maps $\mathbf{v}_i$ to a *different* vector $\mathbf{u}_i$, living in a *different* space ($\mathbb{R}^m$ rather than $\mathbb{R}^n$), scaled by $\sigma_i$. That is why this works for rectangular matrices, where eigenvectors are not even defined.


## Singular Value Decomposition

We can collect the vectors $\mathbf{v}_i$ into a matrix $V$ and the vectors $\mathbf{u}_i$ into a matrix $U$.
$$
A
\begin{bmatrix}
\vert & \vert &   & \vert \\
\mathbf{v}_1   & \mathbf{v}_2  & \dots  & \mathbf{v}_n  \\
\vert & \vert &  & \vert
\end{bmatrix} =
\begin{bmatrix}
\vert & \vert &   & \vert \\
\mathbf{u}_1   & \mathbf{u}_2  & \dots  & \mathbf{u}_n  \\
\vert & \vert &  & \vert
\end{bmatrix}
\left[
\begin{array}{c|c}
\begin{matrix}
\sigma_1 & \cdots & 0 \\
\vdots & \ddots & \vdots \\
0 & \cdots & \sigma_k
\end{matrix}
&
\mathbf{0}
\\
\hline
\mathbf{0} & \mathbf{0}
\end{array}
\right]
.
$$

We call the $\mathbf{v}_i$ the __right singular vectors__ and the $\mathbf{u}_i$ the __left singular vectors__.


## Singular Value Decomposition


And because $V$ is an orthogonal matrix, we have $V V^T = I$, so we can right multiply both sides by $V^T$ to get

$$
A
 =
\begin{bmatrix}
\vert & \vert &   & \vert \\
\mathbf{u}_1   & \mathbf{u}_2  & \dots  & \mathbf{u}_n  \\
\vert & \vert &  & \vert
\end{bmatrix}
\left[
\begin{array}{c|c}
\begin{matrix}
\sigma_1 & \cdots & 0 \\
\vdots & \ddots & \vdots \\
0 & \cdots & \sigma_k
\end{matrix}
&
\mathbf{0}
\\
\hline
\mathbf{0} & \mathbf{0}
\end{array}
\right]
\begin{bmatrix}
\vert & \vert &   & \vert \\
\mathbf{v}_1   & \mathbf{v}_2  & \dots  & \mathbf{v}_n  \\
\vert & \vert &  & \vert
\end{bmatrix}^T.
$$



We can write this as

$$
A = U\Sigma V^{T}.
$$


## Singular Value Decomposition

The SVD of a matrix $A\in\mathbb{R}^{m\times n}$ (where $m>n$) is

$$
A = U\Sigma V^{T},
$$

where

- $U$ has dimension $m\times n$. The columns of $U$ are orthonormal, so $U^TU = I_n$. The columns of $U$ are the __left singular vectors__.
- $\Sigma$ has dimension $n\times n$. The only non-zero values are on the main diagonal and they are nonnegative real numbers  $\sigma_1\geq \sigma_2 \geq \ldots \geq \sigma_k > 0$ and $\sigma_{k+1} = \ldots = \sigma_n = 0$. These are called the __singular values__ of $A$.
- $V$ has dimension $n \times n$. The columns of $V$ are orthonormal, so $V$ is an orthogonal matrix. The columns of $V$ are the __right singular vectors__.


**Thin versus full SVD**

This is the __thin__ (or *economy*) SVD, and it is the one we use throughout. The __full__ SVD pads $U$ with $m-n$ more orthonormal columns to make it a square $m\times m$ orthogonal matrix, and pads $\Sigma$ with $m-n$ rows of zeros to make it $m\times n$. The extra columns of $U$ multiply only zeros, so the two versions give the same $A$. `np.linalg.svd` returns the full version by default; pass `full_matrices=False` to get the thin one, which is what you want for data matrices ($1008\times 1008$ versus $1008\times 121$ for the traffic matrix below).


## SVD Matrix Shapes

In [ ]:
#| fig-align: center
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Create a figure and axis
fig, ax = plt.subplots()

# Draw matrix A
rect_A = patches.Rectangle((0, 0), 2, 3, linewidth=1, edgecolor='black', facecolor='none')
ax.add_patch(rect_A)
ax.text(1, 1.5, r'$A$', fontsize=20, ha='center', va='center')
ax.text(1, -0.5, r'$(m \times n)$', fontsize=12, ha='center', va='center')

# Draw equal sign
ax.text(2.5, 1.5, r'$=$', fontsize=20, ha='center', va='center')

# Draw matrix U
rect_U = patches.Rectangle((3, 0), 2, 3, linewidth=1, edgecolor='black', facecolor='none')
ax.add_patch(rect_U)
ax.text(4, 1.5, r'$U$', fontsize=20, ha='center', va='center')
ax.text(4, -0.5, r'$(m \times n)$', fontsize=12, ha='center', va='center')

# Draw Sigma
rect_Sigma = patches.Rectangle((5.5, 1), 2, 2, linewidth=1, edgecolor='black', facecolor='none')
ax.add_patch(rect_Sigma)
ax.text(6.5, 2, r'$\Sigma$', fontsize=20, ha='center', va='center')
ax.text(6.5, 0.5, r'$(n \times n)$', fontsize=12, ha='center', va='center')

# Draw matrix V^T with the same dimensions as Sigma
rect_VT = patches.Rectangle((8, 1), 2, 2, linewidth=1, edgecolor='black', facecolor='none')
ax.add_patch(rect_VT)
ax.text(9, 2, r'$V^T$', fontsize=20, ha='center', va='center')
ax.text(9, 0.5, r'$(n \times n)$', fontsize=12, ha='center', va='center')

# Set limits and remove axes
ax.set_xlim(-1, 11)
ax.set_ylim(-2, 4)
ax.axis('off')

# Show the plot
plt.show()

## Existence and Uniqueness

<br><br>

* The SVD of a matrix always exists. The next slide shows why.

    * The SVD itself dates to Beltrami and Jordan in the 1870s. That its truncation gives the best low-rank approximation was proven in 1936 by [Carl Eckart and Gale Young](https://www.cambridge.org/core/journals/psychometrika/article/abs/approximation-of-one-matrix-by-another-of-lower-rank/B29672E1EDD0FA1B7611D4DFAFC321B3?utm_campaign=shareaholic&utm_medium=copy_link&utm_source=bookmark).

* The singular values are uniquely determined.

* When the singular values are distinct, the left and right singular vectors are uniquely determined up to sign: replacing $\mathbf{u}_i, \mathbf{v}_i$ by $-\mathbf{u}_i, -\mathbf{v}_i$ leaves $\sigma_i\mathbf{u}_i\mathbf{v}_i^T$ unchanged. (Repeated singular values allow more freedom, as the $\mathbf{u}_i$ beyond the rank already showed.)

## Where do $U$, $\Sigma$, $V$ come from?

Suppose $A = U\Sigma V^T$. Then, using $U^TU = I$,

$$
A^TA = (U\Sigma V^T)^T (U\Sigma V^T) = V \Sigma U^T U \Sigma V^T = V \Sigma^2 V^T.
$$

Look at what this says. $A^TA$ is a square ($n\times n$) __symmetric__ matrix, and $V\Sigma^2V^T$ is exactly the eigendecomposition $Q\Lambda Q^T$ of the previous section:

* the right singular vectors $\mathbf{v}_i$ are the __eigenvectors of $A^TA$__, and
* the squared singular values $\sigma_i^2$ are its __eigenvalues__: $\sigma_i = \sqrt{\lambda_i(A^TA)}$.


Once we have $\mathbf{v}_i$ and $\sigma_i$, the defining relation $A\mathbf{v}_i = \sigma_i\mathbf{u}_i$ hands us the left singular vectors:

$$
\mathbf{u}_i = \frac{A\mathbf{v}_i}{\sigma_i} \qquad (\sigma_i > 0).
$$

Likewise $AA^T = U\Sigma^2U^T$, so the $\mathbf{u}_i$ are the eigenvectors of the $m\times m$ matrix $AA^T$.


---

This is also why the SVD __always exists__, even though $A$ is rectangular and has no eigenvectors of its own:

* $A^TA$ is always square and symmetric, so the spectral theorem gives it real eigenvalues and orthonormal eigenvectors $\mathbf{v}_i$;
* it has the form $B^TB$, so its eigenvalues are $\geq 0$ and we may take square roots to get the $\sigma_i$;
* the $\mathbf{u}_i = A\mathbf{v}_i/\sigma_i$ are automatically orthonormal: $\mathbf{u}_i^T\mathbf{u}_j = \dfrac{\mathbf{v}_i^T A^TA \mathbf{v}_j}{\sigma_i\sigma_j} = \dfrac{\sigma_j^2\, \mathbf{v}_i^T\mathbf{v}_j}{\sigma_i\sigma_j} = \begin{cases}1 & i=j\\ 0 & i\neq j.\end{cases}$


In practice `np.linalg.svd` does __not__ form $A^TA$ (squaring the matrix squares its condition number); it works on $A$ directly. But the recipe above is how you compute a small SVD by hand, and it is the identity that connects the SVD to PCA next lecture.


## A Worked Example

Let's find the SVD of the $3\times 2$ matrix

$$
A = \begin{bmatrix} 1 & 1 \\ 0 & 1 \\ 1 & 0 \end{bmatrix}.
$$

**Step 1: $A^TA$ and its eigendecomposition.**

$$
A^TA = \begin{bmatrix} 1 & 0 & 1 \\ 1 & 1 & 0 \end{bmatrix}\begin{bmatrix} 1 & 1 \\ 0 & 1 \\ 1 & 0 \end{bmatrix} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix},
$$

the matrix from the eigenvalue example: $\lambda_1 = 3$, $\lambda_2 = 1$, with eigenvectors $\mathbf{v}_1 = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}$, $\mathbf{v}_2 = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix}$.


**Step 2: singular values.** $\sigma_1 = \sqrt{3}$, $\sigma_2 = \sqrt{1} = 1$. Both are non-zero, so $A$ has rank 2 (full rank).


---

**Step 3: left singular vectors**, $\mathbf{u}_i = A\mathbf{v}_i/\sigma_i$.

$$
A\mathbf{v}_1 = \frac{1}{\sqrt{2}}\begin{bmatrix} 2 \\ 1 \\ 1 \end{bmatrix}
\;\Rightarrow\;
\mathbf{u}_1 = \frac{1}{\sqrt{3}}\cdot\frac{1}{\sqrt{2}}\begin{bmatrix} 2 \\ 1 \\ 1 \end{bmatrix} = \frac{1}{\sqrt{6}}\begin{bmatrix} 2 \\ 1 \\ 1 \end{bmatrix},
\qquad
A\mathbf{v}_2 = \frac{1}{\sqrt{2}}\begin{bmatrix} 0 \\ -1 \\ 1 \end{bmatrix}
\;\Rightarrow\;
\mathbf{u}_2 = \frac{1}{\sqrt{2}}\begin{bmatrix} 0 \\ -1 \\ 1 \end{bmatrix}.
$$

Check: $\Vert\mathbf{u}_1\Vert^2 = \frac{4+1+1}{6} = 1$, $\Vert\mathbf{u}_2\Vert^2 = \frac{0+1+1}{2} = 1$, and $\mathbf{u}_1^T\mathbf{u}_2 = \frac{0 - 1 + 1}{\sqrt{12}} = 0$.

**Result.**

$$
A = U\Sigma V^T =
\underbrace{\begin{bmatrix} 2/\sqrt{6} & 0 \\ 1/\sqrt{6} & -1/\sqrt{2} \\ 1/\sqrt{6} & 1/\sqrt{2} \end{bmatrix}}_{U\ (3\times 2)}
\underbrace{\begin{bmatrix} \sqrt{3} & 0 \\ 0 & 1 \end{bmatrix}}_{\Sigma\ (2\times 2)}
\underbrace{\frac{1}{\sqrt{2}}\begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}}_{V^T\ (2\times 2)}.
$$


Two sanity checks we will justify shortly: $\Vert A\Vert_F^2 = 1+1+0+1+1+0 = 4 = 3 + 1 = \sigma_1^2 + \sigma_2^2$, and the number of non-zero singular values (2) equals the rank.


---

Let's confirm with `numpy`.

In [ ]:
#| code-fold: false
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[1, 1],
              [0, 1],
              [1, 0]], dtype=float)

u, s, vt = np.linalg.svd(A, full_matrices=False)
print("singular values:", s, "  (sqrt(3) =", np.sqrt(3).round(4), ")")
print("U =\n", u)
print("V^T =\n", vt)
print("U Sigma V^T reproduces A:", np.allclose(u @ np.diag(s) @ vt, A))

In [ ]:
#| code-fold: false
# The same vectors from the eigendecomposition of A^T A
lam, V = np.linalg.eigh(A.T @ A)        # eigh: eigenvalues in ascending order
print("eigenvalues of A^T A:", lam[::-1], " -> sqrt:", np.sqrt(lam[::-1]))
print("eigenvectors (columns):\n", V[:, ::-1])

Note that `numpy` returned $\mathbf{u}_1$ and $\mathbf{v}_1$ with the opposite sign from our hand computation. Both are correct: the singular vectors are only determined up to sign, and flipping $\mathbf{u}_1$ and $\mathbf{v}_1$ together leaves $\sigma_1 \mathbf{u}_1\mathbf{v}_1^T$ unchanged.

## The SVD as a Geometric Picture

$A = U\Sigma V^T$ says that multiplying by $A$ is three simple steps: __rotate__ (by $V^T$), __stretch__ along the axes (by $\Sigma$), __rotate__ again (by $U$).

In [ ]:
#| fig-align: center
#| code-fold: true
#| fig-cap: "Multiplying the unit circle by $A$, one factor at a time. $V^T$ rotates $\\mathbf{v}_1, \\mathbf{v}_2$ onto the coordinate axes, $\\Sigma$ stretches the axes by $\\sigma_1, \\sigma_2$, and $U$ rotates the result so the axes land on $\\mathbf{u}_1, \\mathbf{u}_2$. Unlike an eigenvector, $\\mathbf{v}_i$ *changes direction*: it ends up along $\\mathbf{u}_i$."
A2 = np.array([[3.0, 1.0],
               [0.0, 2.0]])
U2, s2, V2t = np.linalg.svd(A2)
# fix the (arbitrary) sign so that v1 points into the first quadrant
for i in range(2):
    if V2t[i, 0] < 0:
        V2t[i] *= -1
        U2[:, i] *= -1
V2 = V2t.T

theta = np.linspace(0, 2 * np.pi, 400)
circle = np.vstack([np.cos(theta), np.sin(theta)])          # 2 x 400 unit circle
steps = [
    (r'unit circle, $\mathbf{v}_1, \mathbf{v}_2$', circle, V2),
    (r'after $V^T$: rotate', V2t @ circle, V2t @ V2),
    (r'after $\Sigma$: stretch by $\sigma_1, \sigma_2$', np.diag(s2) @ V2t @ circle, np.diag(s2) @ V2t @ V2),
    (r'after $U$: rotate $\;\Rightarrow\; A\mathbf{v}_i = \sigma_i\mathbf{u}_i$', A2 @ circle, A2 @ V2),
]
colors = ['tab:red', 'tab:blue']
labels = [[r'$\mathbf{v}_1$', r'$\mathbf{v}_2$'],
          [r'$\mathbf{e}_1$', r'$\mathbf{e}_2$'],
          [r'$\sigma_1\mathbf{e}_1$', r'$\sigma_2\mathbf{e}_2$'],
          [r'$\sigma_1\mathbf{u}_1$', r'$\sigma_2\mathbf{u}_2$']]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, (title, pts, arrows), lab in zip(axes, steps, labels):
    ax.plot(pts[0], pts[1], color='gray', lw=1.5)
    for j in range(2):
        ax.annotate('', xy=arrows[:, j], xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=colors[j], lw=2))
        ax.text(*(arrows[:, j] * 1.15), lab[j], color=colors[j], fontsize=12,
                ha='center', va='center')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlim(-4.2, 4.2); ax.set_ylim(-4.2, 4.2)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=11)
fig.suptitle('A = [[3, 1], [0, 2]]:   ' + r'$\sigma_1 = %.2f,\ \sigma_2 = %.2f$' % tuple(s2),
             fontsize=13)
plt.tight_layout()
plt.show()

The image of the unit circle is always an ellipse whose semi-axes have lengths $\sigma_1 \geq \sigma_2$ and point along $\mathbf{u}_1, \mathbf{u}_2$. The singular values therefore measure how much $A$ stretches space in its most and least sensitive directions. For $m\times n$ matrices the same picture holds with an ellipsoid in $\mathbb{R}^m$; directions $\mathbf{v}_i$ with $\sigma_i = 0$ are flattened out entirely.


## Outer Products

The SVD can also be represented as a sum of outer products

$$
A = \sum_{i=1}^{n} \sigma_{i}\mathbf{u}_i\mathbf{v}_{i}^{T},
$$

where $\mathbf{u}_i, \mathbf{v}_{i}$ are the $i$-th columns of $U$ and $V$, respectively.

Only the first $k$ terms are non-zero, since $\sigma_{k+1} = \dots = \sigma_n = 0$. In other words, **the rank of $A$ is the number of non-zero singular values**.

On a computer the singular values of a rank-deficient matrix are rarely *exactly* zero; `np.linalg.matrix_rank` counts the singular values above a small tolerance (by default $\sigma_1 \cdot \max(m,n) \cdot \varepsilon_{\text{machine}}$).


## Outer Products, continued

An outer product of a $m\times 1$ vector and a $1\times n$ vector is a $m\times n$ matrix.

$$\mathbf{u}_i\mathbf{v}_{i}^{T}=
\begin{bmatrix}
u_{i1} \\
u_{i2} \\
\vdots \\
u_{im} \\
\end{bmatrix}
\begin{bmatrix}
v_{i1} & v_{i2} & \cdots & v_{in} \\
\end{bmatrix} =
\begin{bmatrix}
u_{i1}v_{i1} & u_{i1}v_{i2} & \cdots & u_{i1}v_{in} \\
u_{i2}v_{i1} & u_{i2}v_{i2}  & \cdots & u_{i2}v_{in} \\
\vdots & \vdots & \ddots & \vdots \\
u_{im}v_{i1} &  u_{im}v_{i2}  & \cdots & u_{im}v_{in} \\
\end{bmatrix}
$$

Here $u_{ij}$ denotes the $j$-th entry of $\mathbf{u}_i$, and likewise $v_{ij}$ is the $j$-th entry of $\mathbf{v}_i$.

It is a rank-1 matrix.  How can you tell?

**Alternate Interpretation:**

The SVD decomposes $A$ into a linear combination of rank-1 matrices.

The singular value tells us the weight (contribution) of each rank-1 matrix to the matrix $A$.


---

Here is what that looks like for a photograph (a $512\times 512$ matrix of gray levels that we will come back to later in the lecture).

![](figs/L12-recap-boat-layers.png)

Top row: individual rank-1 layers $\sigma_i\mathbf{u}_i\mathbf{v}_i^T$ — each is a *row profile* times a *column profile*, so it can only contain horizontal and vertical stripes. Bottom row: partial sums $A^{(k)} = \sum_{i \leq k}\sigma_i\mathbf{u}_i\mathbf{v}_i^T$. Ten layers already give the boats; the remaining 502 add texture.

# Low Rank Approximations

## Low Effective Rank

In many situations we want to __approximate__ a matrix $A$ with a low-rank matrix $A^{(k)}.$

To talk about when one matrix *approximates* another, we need a norm for matrices.


We will use the __Frobenius norm__, which is defined as

$$\Vert A\Vert_F = \sqrt{\sum_{i,j} a_{ij}^2}.$$


Observe that this is the $\ell_2$ norm for a vectorized matrix, i.e., by  stacking the columns of the matrix $A$ to form a vector of length $mn$.


The Frobenius norm is tied to the singular values by

$$\Vert A\Vert_F^2 = \sum_{i=1}^{n} \sigma_i^2,$$

because multiplying by the orthogonal $U$ and $V^T$ does not change lengths, so $A = U\Sigma V^T$ has the same Frobenius norm as $\Sigma$. (Check it on the worked example: $4 = 3 + 1$.)


---

To quantify when one matrix is *close* to another, we define the distance function:

$$
\operatorname{dist}(A,B) = \Vert A-B\Vert_F.
$$

This can be viewed as Euclidean distance between $mn$-dimensional vectors.

We define the optimal __rank-$k$ approximation__ to $A$ as

$$
A^{(k)} =\mathop{\arg\min}\limits_{\{B~|~\operatorname{Rank} B = k\}} \Vert A-B\Vert_F.
$$

In other words, $A^{(k)}$ is the closest rank-$k$ matrix to $A$.

## Finding Rank-$k$ Approximations

How can we find the optimal rank-$k$ approximation to a matrix $A$?

The __Singular Value Decomposition (SVD).__


Why?


The SVD  gives the best rank-$k$ approximation to $A$ for __every__ $k$ up to the rank of $A$.


---

To form the best rank-$k$ approximation to $A$ using the SVD you calculate

$$ A^{(k)} = U'\Sigma'(V')^T,$$

where

* $U'$ are the $k$ leftmost columns of $U$,
* $\Sigma'$ is the $k\times k$ upper left sub-matrix of $\Sigma$, and
* $V'$ is the $k$ leftmost columns of $V$ (or the $k$ topmost rows of $V^T$)


Equivalently, keep the first $k$ terms of the outer-product sum: $A^{(k)} = \sum_{i=1}^{k} \sigma_i\mathbf{u}_i\mathbf{v}_i^T$.


---

SVD (all $n$ terms):

$$
A =
\begin{bmatrix}
\vert & \vert &   & \vert \\
\mathbf{u}_1   & \mathbf{u}_2  & \dots  & \mathbf{u}_n  \\
\vert & \vert &  & \vert
\end{bmatrix}
\left[
\begin{array}{c|c}
\begin{matrix}
\sigma_1 & \cdots & 0 \\
\vdots & \ddots & \vdots \\
0 & \cdots & \sigma_k
\end{matrix}
&
\mathbf{0}
\\
\hline
\mathbf{0} & \mathbf{0}
\end{array}
\right]
\begin{bmatrix}
- & \mathbf{v}_1^T & - \\
- & \mathbf{v}_2^T & - \\
  & \vdots &   \\
- & \mathbf{v}_n^T & -
\end{bmatrix}
$$

Rank-$k$ SVD (first $k$ terms):

$$
A^{(k)} =
\begin{bmatrix}
\vert & \vert &   & \vert \\
\mathbf{u}_1   & \mathbf{u}_2  & \dots  & \mathbf{u}_k  \\
\vert & \vert &  & \vert
\end{bmatrix}
\begin{bmatrix}
\sigma_1 & \cdots & 0 \\
\vdots & \ddots & \vdots \\
0 & \cdots & \sigma_k
\end{bmatrix}
\begin{bmatrix}
- & \mathbf{v}_1^T & - \\
- & \mathbf{v}_2^T & - \\
  & \vdots &   \\
- & \mathbf{v}_k^T & -
\end{bmatrix}
$$

## The Eckart–Young Theorem

**Eckart–Young theorem (1936)**

Let $A = U\Sigma V^T$ have singular values $\sigma_1 \geq \sigma_2 \geq \dots \geq \sigma_n$. Among __all__ matrices $B$ of rank at most $k$, the truncated SVD $A^{(k)} = U'\Sigma'(V')^T$ minimizes $\Vert A - B\Vert_F$, and

$$\Vert A-A^{(k)}\Vert_F^2 = \sum_{i=k+1}^n\sigma^2_i.$$


This means that the distance (in Frobenius norm) of the best rank-$k$ approximation $A^{(k)}$ from $A$ is equal to $\sqrt{\sum_{i=k+1}^n\sigma_i^2}$: the discarded singular values *are* the error.

Dividing by $\Vert A\Vert_F^2 = \sum_i \sigma_i^2$ gives the __relative error__, which we can read off the singular values without ever forming $A^{(k)}$:

$$
\frac{\Vert A-A^{(k)}\Vert_F^2}{\Vert A\Vert_F^2} = \frac{\sum_{i>k}\sigma_i^2}{\sum_{i}\sigma_i^2}.
$$


__In the worked example:__ $A^{(1)} = \sigma_1\mathbf{u}_1\mathbf{v}_1^T = \sqrt{3}\cdot\frac{1}{\sqrt{6}}\begin{bmatrix}2\\1\\1\end{bmatrix}\frac{1}{\sqrt{2}}\begin{bmatrix}1 & 1\end{bmatrix} = \begin{bmatrix} 1 & 1 \\ 0.5 & 0.5 \\ 0.5 & 0.5 \end{bmatrix}$, so $A - A^{(1)} = \begin{bmatrix} 0 & 0 \\ -0.5 & 0.5 \\ 0.5 & -0.5 \end{bmatrix}$ and $\Vert A - A^{(1)}\Vert_F = \sqrt{4 \cdot 0.25} = 1 = \sigma_2$. No rank-1 matrix is closer to $A$.


# Low Rank Approximations in Practice

## Low Effective Rank of Data Matrices

In general, a data matrix $A\in\mathbb{R}^{m\times n}$  is usually __full rank__, meaning that $\operatorname{Rank}(A) = \min(m, n)$.

However, it is possible to encounter data matrices that have __low effective rank__.


This means that we can approximate $A$ by some $A^{(k)}$ for which $k \ll \min(m,n)$.


For any data matrix, we can judge when this is the case by looking at its singular values, because the singular values tell us the distance to the nearest rank-$k$ matrix.


## Traffic Data

Let's see how this theory can be used in practice  and investigate some real data.

We'll look at data traffic on the Abilene network:

![](figs/L10-Abilene-map.png)

Source: Internet2, circa 2005

---

Each row is a 10-minute interval over one week (1008 rows); each column is one origin–destination flow between two of the eleven Abilene routers (121 columns).

In [ ]:
#| code-fold: false
import pandas as pd

with open('data/net-traffic/AbileneFlows/odnames','r') as f:
    odnames = [line.strip() for line in f]
dates = pd.date_range('9/1/2003', freq = '10min', periods = 1008)
Atraf = pd.read_table('data/net-traffic/AbileneFlows/X', sep='  ', header=None, names=odnames, engine='python')
Atraf.index = dates
Atraf.shape

In [ ]:
Atraf.head(4)

---

As we would expect, our traffic matrix has rank 121:

In [ ]:
#| code-fold: false
np.linalg.matrix_rank(Atraf)

However -- perhaps it has low __effective__ rank.

The `numpy` routine for computing the SVD is `np.linalg.svd`. We ask for the thin SVD:

In [ ]:
#| code-fold: false
u, s, vt = np.linalg.svd(Atraf, full_matrices=False)
u.shape, s.shape, vt.shape

---

Now let's look at the singular values of `Atraf` to see if it can be usefully approximated as a low-rank matrix:

In [ ]:
#| fig-align: center
fig = plt.figure(figsize=(5, 3))
plt.plot(range(1, 1+len(s)), s)
plt.xlabel(r'$k$', size=20)
plt.ylabel(r'$\sigma_k$', size=20)
plt.ylim(ymin=0)
plt.xlim(xmin=-1)
plt.title(r'Singular Values of $A$', size=20)
plt.show()

This classic, sharp-elbow tells us that a few singular values are very large, and most singular values are quite small. In fact $\sigma_1^2$ alone is 95% of $\sum_i\sigma_i^2 = \Vert A\Vert_F^2$.

---

Zooming in for just small $k$ values, and scaling by $\Vert A\Vert_F$, we can see that the elbow is around 4 - 6 singular values:

In [ ]:
#| fig-align: center
fig = plt.figure(figsize=(5, 3))
Anorm = np.linalg.norm(Atraf)
plt.plot(range(1, 21), s[0:20]/Anorm, '.-')
plt.xlim([0.5, 20])
plt.ylim([0, 1])
plt.xlabel(r'$k$', size=20)
plt.xticks(range(1, 21))
plt.ylabel(r'$\sigma_k\,/\,\Vert A\Vert_F$', size=16);
plt.title(r'Singular Values of $A$',size=20)
plt.show()

This pattern of singular values suggests __low effective rank.__

---

Let's use the Eckart–Young formula to compute the relative error of a rank-$k$ approximation to $A$, for every $k$ at once:

In [ ]:
#| fig-align: center
fig = plt.figure(figsize=(5, 3))
Anorm = np.linalg.norm(Atraf)                  # = sqrt(sum of s**2)
err = np.sqrt(np.cumsum(s[::-1]**2)[::-1])     # err[k] = sqrt(sum_{i>k} s_i^2), k = 0, 1, ...
plt.plot(range(1, 21), err[1:21]/Anorm, '.-')
plt.xlim([0, 21])
plt.ylim([0, 1])
plt.xticks(range(1, 21))
plt.xlabel(r'$k$', size = 16)
plt.ylabel(r'relative F-norm error', size=16)
plt.title(r'Relative Error of rank-$k$ approximation to $A$', size=16)
plt.show()

Remarkably, we are down to 9% relative error using only a rank 20 approximation to $A$.

---

So instead of storing

* $mn =$ (1008 $\cdot$ 121) = 121,968 values,

we only need to store

* $k(m+n)$ = 20 $\cdot$ (1008 + 121) = 22,580 values,

which is an 81% reduction in size.

## Occam's Razor

![](figs/L10-William-of-Ockham.png)

[Source](https://commons.wikimedia.org/w/index.php?curid=5523066)

Why should a week of traffic on 121 links be described so well by 20 patterns?

A model is a description of the data that is *simpler* than the data, and modeling works surprisingly often. William of Ockham (c. 1300) put it this way:

> Entities must not be multiplied beyond necessity.

that is, among competing hypotheses, prefer the one with the fewest assumptions.


"Occam's razor" is an observation about the world: it is more common for a set of observations to be generated by a simple process than a complex one. The world is full of simple, but often hidden, patterns — and low effective rank is one precise, measurable form of that simplicity.



## Low Effective Rank is Common

In practice __many__ datasets have low effective rank.

We consider the following examples:

* Likes on Facebook, Yelp reviews and Tweets (the site formerly known as Twitter),
* User preferences over time,
* Images.


## Social Media Activity

Here, the matrices are Users $\times$ Categories:

1. Number of Facebook likes:  Users $\times$ Page Categories
2. Number of Yelp reviews:  Users $\times$ Yelp Categories
3. Number of Tweets:  Users $\times$ Topic Categories


![](figs/L12-social-media-spatial-spectra.png)

Source: [Viswanath et al., Usenix Security, 2014]


The axes are the paper's: the "fraction of variance" of principal component $i$ is $\sigma_i^2/\sum_j\sigma_j^2$ for the *centered* matrix — that is PCA, next lecture's topic, but it is the same spectrum-with-an-elbow we just saw. These user $\times$ category matrices are strongly low rank: a handful of components carry most of the variance. (The same paper's user $\times$ time-bin matrices are *not* — their top component explains only 2–13% — so low effective rank is common, not universal.)



## Netflix

Example: the Netflix prize worked with partially-observed (sparse) rating matrices like this:



$$
\begin{bmatrix}
 & & & \vdots & & & \\
 & & 3 & 2 & & 1 &\\
 & 1 & & 1 & & & \\
\dots & & 2 & & 4 & & \dots\\
 & 5 & 5 & & 4 & & \\
 & 1 & & & 1 & 5 & \\
 & & & \vdots & & & \\
\end{bmatrix},
$$




where the rows correspond to users, the columns to movies, and the entries are ratings.


Although the problem matrix was of size 500,000 $\times$ 18,000, the winning approach modeled the matrix as having __rank 20 to 40.__

Source: [Koren et al, IEEE Computer, 2009]



**Not quite the SVD**

The Netflix matrix is about 99% empty, and the SVD needs every entry. The winning entry instead used matrix *factorization*: a rank-$k$ model $\tilde U\tilde V$ fitted only to the observed ratings. Same low-rank idea, different algorithm — it is the second half of Lecture 19.


## Images

Image data often shows low effective rank.

For example, here is an original $512 \times 512$ photo:

In [ ]:
#| fig-align: center
boat = np.loadtxt('data/images/boat/boat.dat')
import matplotlib.cm as cm
plt.figure()
plt.imshow(boat, cmap=cm.Greys_r)
plt.axis('off')
plt.show()

---

Treat the image as a matrix and then compute the singular values. They span five orders of magnitude, so we use a log scale:

In [ ]:
#| fig-align: center
u, s, vt = np.linalg.svd(boat, full_matrices=False)
plt.semilogy(range(1, 1+len(s)), s)
plt.xlabel('$k$', size=16)
plt.ylabel(r'$\sigma_k$ (log scale)', size=16)
plt.title('Singular Values of Boat Image', size=16)
plt.show()

---

This image is 512 $\times$ 512. As a matrix, it has rank of 512.

But its _effective_ rank is low: $\sigma_1$ is eight times $\sigma_2$, and by $k = 40$ the singular values are down to about 1% of $\sigma_1$.

The Eckart–Young formula turns that into an error curve:

In [ ]:
#| fig-align: center
Bnorm = np.linalg.norm(boat)
err = np.sqrt(np.cumsum(s[::-1]**2)[::-1])
ks = range(1, 101)
plt.figure(figsize=(6, 3.5))
plt.plot(ks, err[1:101]/Bnorm, '-')
plt.plot(40, err[40]/Bnorm, 'o', color='tab:red')
plt.annotate(f'k = 40: {err[40]/Bnorm:.1%}', xy=(40, err[40]/Bnorm),
             xytext=(50, 0.2), arrowprops=dict(arrowstyle='->'))
plt.ylim([0, 0.4])
plt.xlabel(r'$k$', size=16)
plt.ylabel('relative F-norm error', size=14)
plt.title(r'Relative Error of rank-$k$ approximation to the image', size=14)
plt.show()

---

At rank 40 the relative error is 7.9%, and we store $k(m + n + 1) = 40\cdot(512 + 512 + 1) = 41{,}000$ numbers instead of $512^2 = 262{,}144$ — a $6.4\times$ compression.

Let's find the closest rank-40 matrix and view it.

In [ ]:
#| code-fold: false
k = 40
boatApprox = u[:, :k] @ np.diag(s[:k]) @ vt[:k]

In [ ]:
#| fig-align: center
plt.figure(figsize=(9, 6))
plt.subplot(1, 2, 1)
plt.imshow(boatApprox, cmap=cm.Greys_r)
plt.axis('off')
plt.title('Rank 40 Boat')
plt.subplot(1, 2, 2)
plt.imshow(boat, cmap=cm.Greys_r)
plt.axis('off')
plt.title('Full Rank 512 Boat')
plt.show()

## Interpretations of Low Effective Rank

How can we understand the low-effective-rank phenomenon in general?

There are two helpful interpretations:

1. Common Patterns
2. Latent Factors


## Low Rank Implies Common Patterns

The first interpretation of low-rank behavior is in answering the question:

"What is the strongest pattern in the data?"


Remember that using the SVD we form the low-rank approximation as

$$ A^{(k)} =  U'\Sigma'(V')^T$$

and


- $U'$ are the $k$ leftmost columns of $U$,
- $\Sigma'$ is the $k\times k$ upper left submatrix of $\Sigma$, and
- $V'$ are the $k$ leftmost columns of $V$.


In this interpretation, we think of each column of $A^{(k)}$ as a combination of the columns of $U'$.


How can this be helpful?



## Common Patterns: Traffic Example

Consider the set of traffic traces. There are clearly some common patterns. How can we find them?

In [ ]:
#| fig-align: center
plt.figure(figsize=(10, 8))
for i in range(12):
    ax = plt.subplot(4, 3, i + 1)
    Atraf.iloc[:, i].plot()
    plt.title(odnames[i])
plt.subplots_adjust(hspace=1)
plt.suptitle('The First Twelve Traffic Traces', size=20)
plt.show()

---

Let's use as our example the ATLA-CHIN flow — traffic from Atlanta to Chicago. It is the second column of $A$ (the first, ATLA-ATLA, is traffic that enters and leaves at Atlanta):

In [ ]:
#| code-fold: false
j = odnames.index('ATLA-CHIN')
j

The outer-product form of the SVD tells us that column $j$ is

$$\mathbf{a}_j \approx \sigma_1 v_{1j}\mathbf{u}_1 + \sigma_2 v_{2j}\mathbf{u}_2 + \dots + \sigma_k v_{kj}\mathbf{u}_k,$$

where $v_{ij}$ is the $j$-th entry of $\mathbf{v}_i$ — so the coefficients are the $j$-th column of $V^T$.

Every one of the 121 traces is a combination of the __same__ $k$ vectors $\mathbf{u}_1, \ldots, \mathbf{u}_k$; only the coefficients change from trace to trace. In other words, $\mathbf{u}_1$ (the first column of $U$) is the "strongest" pattern occurring in $A$, and its strength is measured by $\sigma_1$.

---

Here is a view of the first 2 columns of $U\Sigma$, i.e. $\sigma_1\mathbf{u}_1$ and $\sigma_2\mathbf{u}_2$, for the traffic matrix data.

These are the strongest patterns occurring across all of the 121 traces.

In [ ]:
#| fig-align: center
u, s, vt = np.linalg.svd(Atraf, full_matrices=False)

# Singular vectors are only determined up to sign, and LAPACK happened to
# return u_1 entirely negative. Flip u_1 and v_1 together (which leaves
# sigma_1 u_1 v_1^T unchanged) so the pattern reads as traffic volume.
sign = np.sign(u[:, 0].sum())
u[:, 0] *= sign
vt[0] *= sign

uframe = pd.DataFrame(u @ np.diag(s), index=dates)
uframe[0].plot(color='r', label=r'$\sigma_1 \mathbf{u}_1$')
uframe[1].plot(label=r'$\sigma_2 \mathbf{u}_2$')
plt.legend(loc='best')
plt.ylabel('scaled pattern (traffic units)')
plt.title(r'First Two Columns of $U\Sigma$')
plt.show()

The first pattern is the daily rhythm shared by every flow: low overnight, high during the day, weaker at the weekend. Each trace, ATLA-CHIN included, is mostly a rescaled copy of it — which is exactly what $\sigma_1^2$ being 95% of $\Vert A\Vert_F^2$ means.

## Low Rank Defines Latent Factors

The next interpretation of low-rank behavior is that it exposes "latent factors" that describe the data.

In this interpretation, we think of each element of $A^{(k)}=U'\Sigma'(V')^T$ as the inner product of a row of $U'\Sigma'$ and a column of $(V')^{T}$ (equivalently a row of $V'$).


Let's say we are working with a matrix of users and items.


In particular, let the items be movies and matrix entries be ratings, as in the Netflix prize.


## Latent Factors: Netflix example

Recall the structure from earlier:

$$
\begin{bmatrix}
\vdots & \vdots &  & \vdots \\
\mathbf{a}_{1} & \mathbf{a}_{2} & \cdots & \mathbf{a}_{n} \\
\vdots & \vdots &  & \vdots \\
\end{bmatrix}
\approx
\underbrace{
\begin{bmatrix}
\vdots &  \vdots  \\
\sigma_1 \mathbf{u}_1 & \sigma_k \mathbf{u}_{k} \\
\vdots& \vdots  \\
\end{bmatrix}
}_{\tilde{U}\in\mathbb{R}^{m\times k}}
\underbrace{
\begin{bmatrix}
\cdots & \mathbf{v}_{1}^{T} & \cdots   \\
\cdots & \mathbf{v}_{k}^{T} & \cdots   \\
\end{bmatrix}
}_{\tilde{V}\in\mathbb{R}^{k\times n}},
$$

where the rows of $A$ are the users and the columns are movie ratings.

Then the rating that a user gives a movie is the inner product of a $k$-element vector that corresponds to the user, and a $k$-element vector that corresponds to the movie.

In other words we have:

$$
a_{ij} \approx \sum_{\ell=1}^{k} \tilde{U}_{i\ell} \tilde{V}_{\ell j}.
$$

---

We can think of user $i$'s preferences as being captured by row $i$ of $\tilde{U}$, which is a point in $\mathbb{R}^k$.

We have described everything we need to know to predict user $i$'s ratings via a $k$-element vector.


The $k$-element vector is called a __latent factor.__


Likewise, we can think of column $j$ of $\tilde{V}$ as a "description" of movie $j$ (another latent factor).


The value in using latent factors comes from the summarization of user preferences, and the predictive power one obtains.


For example, the winning entry in the Netflix prize competition modeled user preferences with 20 latent factors.


The remarkable thing is that a person's preferences for all 18,000 movies can be reasonably well captured in a vector of dimension 20!


---

Here is a figure from the paper that described the winning strategy in the Netflix prize.

It shows a hypothetical __latent space__ in which each user, and each movie, is represented by a latent vector.

![](figs/L10-Movie-Latent-Space.png)

Source: Koren et al, IEEE Computer, 2009

In practice, this is perhaps a 20- or 40-dimensional space.

---

Here are some representations of movies in that space (reduced to 2-D).

Notice how the space seems to capture similarity among movies!

![](figs/L10-Netflix-Latent-Factors.png)

Source: Koren et al, IEEE Computer, 2009

## Summary

* The SVD $A = U\Sigma V^T$ exists for every matrix: the $\mathbf{v}_i$ are the eigenvectors of the symmetric matrix $A^TA$, $\sigma_i^2$ are its eigenvalues, and $\mathbf{u}_i = A\mathbf{v}_i/\sigma_i$.
* When we are working with data matrices, it is valuable to consider the __effective rank__.
* Many (many) datasets in real life show __low effective rank__.
* This property can be explored precisely using the Singular Value Decomposition of the matrix: by the Eckart–Young theorem, the truncated SVD is the best rank-$k$ approximation, and its error is the sum of the discarded $\sigma_i^2$.
* When low effective rank is present
    * the matrix can be compressed with only small loss of accuracy,
    * we can extract the *strongest* patterns in the data,
    * we can describe each data item in terms of the inner product of __latent factors__.
* __Next lecture:__ center the columns of a data matrix and take its SVD. The columns of $V$ are the *principal directions* and $\sigma_i^2/(m-1)$ is the variance along each — the $A^TA = V\Sigma^2V^T$ identity above *is* PCA.
